# Batch Backtests for Cointegrated Pairs
Use this notebook after running the pair-finder to load its saved candidates, instantiate the Alpaca data handler one time, and iterate through each pair with PairsTradeStrategy.run_backtest().

In [5]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
CANDIDATE_FILE = DATA_DIR / 'cointegrated_pairs.csv'  # update if you save elsewhere

START_DATE = '2005-01-01'
END_DATE = '2010-01-01'
TIMEFRAME = '1H'  # use 4H/1D/etc. if Alpaca access is limited

ENTRY_THRESHOLD = 1.2
EXIT_THRESHOLD = 0.2
LOOKBACK_DAYS = 60
STAT_REFRESH_DAYS = 7
QUANTITY = 10
BENCHMARK_SYMBOL = 'SPY'
INITIAL_CAPITAL = 15_000
MAX_PAIRS = 30

print(f'Project root: {PROJECT_ROOT}')
print(f'Candidate file: {CANDIDATE_FILE}')

Project root: C:\Users\trash\trading-bot
Candidate file: C:\Users\trash\trading-bot\data\cointegrated_pairs.csv


In [6]:
if not CANDIDATE_FILE.exists():
    raise FileNotFoundError(
        f'Candidate file not found at {CANDIDATE_FILE}. Run the pair-finder notebook or update CANDIDATE_FILE.'
    )

candidates_df = pd.read_csv(CANDIDATE_FILE)
candidates_df = candidates_df.rename(columns=str.lower)
required_cols = {'symbol_a', 'symbol_b', 'hedge_ratio'}
missing = required_cols - set(candidates_df.columns)
if missing:
    raise ValueError(f'Missing required columns in candidate file: {missing}')

sort_cols = [col for col in ['p_value', 'z_score_std', 'spread_std'] if col in candidates_df.columns]
if sort_cols:
    candidates_df = candidates_df.sort_values(sort_cols)

candidates_df = candidates_df.reset_index(drop=True)
candidates_df = candidates_df.head(MAX_PAIRS).copy()

display_cols = [col for col in ['symbol_a', 'symbol_b', 'hedge_ratio', 'p_value'] if col in candidates_df.columns]
if display_cols:
    display(candidates_df[display_cols].head(10))
else:
    display(candidates_df.head(10))
print(f'Total candidate pairs in CSV: {len(pd.read_csv(CANDIDATE_FILE))}')
print(f'Testing top {len(candidates_df)} pairs after sorting by {sort_cols or "default order"}.')

,symbol_a,symbol_b,hedge_ratio,p_value
0,V,MA,0.527159,0.008333
1,SPGI,MCO,1.102522,0.023248
2,BLK,USB,9.070770,0.027424
3,MCO,V,1.659694,0.039320


Total candidate pairs in CSV: 4
Testing top 4 pairs after sorting by ['p_value'].


In [7]:
from src.data_handler import DataHandler
from src.strategies.pairs_trade import PairsTradeStrategy

class NoOpExecutionHandler:
    """Placeholder execution handler so the strategy can be instantiated."""
    def execute_order(self, signal):
        print(f'[NoOpExecution] {signal}')

execution_handler = NoOpExecutionHandler()
data_handler = DataHandler(paper_trading=True)

benchmark_series = pd.Series(dtype=float)
benchmark_return = None
if BENCHMARK_SYMBOL:
    benchmark_data = data_handler.get_historical_bars(
        [BENCHMARK_SYMBOL], TIMEFRAME, start=START_DATE, end=END_DATE
    )
    bench_df = benchmark_data.get(BENCHMARK_SYMBOL) if benchmark_data else None
    if bench_df is not None and not bench_df.empty:
        bench_df = bench_df.copy()
        bench_df.index = pd.to_datetime(bench_df.index)
        if getattr(bench_df.index, 'tz', None) is not None:
            bench_df.index = bench_df.index.tz_convert('UTC').tz_localize(None)
        benchmark_series = bench_df['close'].astype(float)
        benchmark_return = float(benchmark_series.iloc[-1] / benchmark_series.iloc[0] - 1)
        print(
            f"Benchmark {BENCHMARK_SYMBOL}: fetched {len(benchmark_series)} bars; "
            f"buy/hold return {benchmark_return:.2%}"
        )
    else:
        print(f'Benchmark data unavailable for {BENCHMARK_SYMBOL}; comparisons disabled.')
else:
    print('BENCHMARK_SYMBOL not set; skipping benchmark fetch.')

print('Handlers ready.')

Data Handler (alpaca-py) initialized.
Submitting data request to Alpaca...
BarSet was returned, but its DataFrame is empty.
Benchmark data unavailable for SPY; comparisons disabled.
Handlers ready.
BarSet was returned, but its DataFrame is empty.
Benchmark data unavailable for SPY; comparisons disabled.
Handlers ready.


In [8]:
backtest_rows = []
for _, row in candidates_df.iterrows():
    symbol_a = row['symbol_a']
    symbol_b = row['symbol_b']
    hedge_ratio = row['hedge_ratio']
    print(f'Running backtest for {symbol_a} / {symbol_b} (hedge_ratio={hedge_ratio:.4f})')
    try:
        strategy = PairsTradeStrategy(
            data_handler=data_handler,
            execution_handler=execution_handler,
            symbol_a=symbol_a,
            symbol_b=symbol_b,
            hedge_ratio=hedge_ratio,
            lookback_days=LOOKBACK_DAYS,
            entry_threshold=ENTRY_THRESHOLD,
            exit_threshold=EXIT_THRESHOLD,
            quantity=QUANTITY,
            timeframe=TIMEFRAME,
            auto_execute=False,
            stat_refresh_days=STAT_REFRESH_DAYS,
            benchmark_symbol=BENCHMARK_SYMBOL,
            initial_capital=INITIAL_CAPITAL,
        )
        summary = strategy.run_backtest(
            start_date=START_DATE,
            end_date=END_DATE,
            timeframe=TIMEFRAME,
            benchmark_series=benchmark_series,
            benchmark_return=benchmark_return,
        )
        backtest_rows.append({
            'symbol_a': symbol_a,
            'symbol_b': symbol_b,
            'hedge_ratio': hedge_ratio,
            'num_trades': summary['num_trades'],
            'open_trades': summary['open_trades'],
            'total_pnl': summary['total_pnl'],
            'win_rate': summary['win_rate'],
            'strategy_return': summary['strategy_return'],
            'benchmark_return': summary['benchmark_return'],
            'excess_return': summary['excess_return'],
            'sharpe': summary['sharpe'],
            'max_drawdown': summary['max_drawdown'],
            'status': 'ok',
        })
    except Exception as exc:
        backtest_rows.append({
            'symbol_a': symbol_a,
            'symbol_b': symbol_b,
            'hedge_ratio': hedge_ratio,
            'status': f'error: {exc}',
        })
        print(f'Failed backtest for {symbol_a}/{symbol_b}: {exc}')

backtest_df = pd.DataFrame(backtest_rows)
success_df = backtest_df[backtest_df['status'] == 'ok'].copy()
if not success_df.empty and 'strategy_return' in success_df:
    success_df = success_df[success_df['benchmark_return'].notna()]
    outperformers = success_df[success_df['strategy_return'] > success_df['benchmark_return']]
    outperformers = outperformers.sort_values(['strategy_return', 'sharpe'], ascending=[False, False])
else:
    outperformers = pd.DataFrame()

print(f"Completed {len(backtest_df)} backtests; {len(outperformers)} pairs beat {BENCHMARK_SYMBOL}.")
outperformers.reset_index(drop=True, inplace=True)
outperformers

Running backtest for V / MA (hedge_ratio=0.5272)
BaseStrategy initialized with symbols: ['V', 'MA']
Submitting data request to Alpaca...
PairsTradeStrategy stats initialised: mean=44.9127, std=4.3336 (window ~422 bars)
Submitting data request to Alpaca...
BarSet was returned, but its DataFrame is empty.
Failed backtest for V/MA: No historical data returned for backtest.
Running backtest for SPGI / MCO (hedge_ratio=1.1025)
BaseStrategy initialized with symbols: ['SPGI', 'MCO']
Submitting data request to Alpaca...
PairsTradeStrategy stats initialised: mean=44.9127, std=4.3336 (window ~422 bars)
Submitting data request to Alpaca...
BarSet was returned, but its DataFrame is empty.
Failed backtest for V/MA: No historical data returned for backtest.
Running backtest for SPGI / MCO (hedge_ratio=1.1025)
BaseStrategy initialized with symbols: ['SPGI', 'MCO']
Submitting data request to Alpaca...
PairsTradeStrategy stats initialised: mean=-43.5188, std=7.7805 (window ~351 bars)
Submitting data re

""
